# Elise Distillation

Distill Elise-264-GSTP into a smaller DQN student.

In [ ]:
# cell: colab-setup
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")
sys.path.insert(0, "distillation/src")

from hpo.environments.solar_system_lander.env import World

from distillation import (
    collect_teacher_dataset,
    collect_teacher_dataset_parallel,
    evaluate_student,
    evaluate_teacher,
    plot_score_gaps,
    plot_score_quantiles,
    score_comparison_table,
    train_student,
)

In [ ]:
# cell: distillation-settings
WORLD_MIX = {World.MERCURY: 1, World.VENUS: 4, World.EARTH: 4, World.MOON: 1, World.MARS: 1}
SEEDS = list(range(1000, 1200)) + [7, 42, 1911, 10014]
EPSILON = 0.05
HIDDEN_SIZES = (64, 64)
EVAL_EPISODES_PER_WORLD = 100
MIN_SCORE_DIFF = 25.0

In [ ]:
# cell: collect-teacher-dataset; requires: colab-setup, distillation-settings
# For faster collection, switch to collect_teacher_dataset_parallel(..., num_envs=16).
dataset = collect_teacher_dataset(epsilon=EPSILON, seeds=SEEDS, world_mix=WORLD_MIX)
dataset.metadata

In [ ]:
# cell: train-student; requires: collect-teacher-dataset
student = train_student(dataset, hidden_sizes=HIDDEN_SIZES)
student.metadata

In [ ]:
# cell: evaluate-student; requires: train-student
student_summary = evaluate_student(student, eval_episodes_per_world=EVAL_EPISODES_PER_WORLD)
student_summary

In [ ]:
# cell: evaluate-teacher-baseline; requires: train-student
teacher_summary = evaluate_teacher(eval_episodes_per_world=EVAL_EPISODES_PER_WORLD)
teacher_summary

In [ ]:
# cell: compare-score-distributions; requires: evaluate-student, evaluate-teacher-baseline
plot_score_quantiles(teacher_summary, student_summary)

In [ ]:
# cell: compare-score-gaps; requires: evaluate-student, evaluate-teacher-baseline
plot_score_gaps(teacher_summary, student_summary)

In [ ]:
# cell: compare-score-table; requires: evaluate-student, evaluate-teacher-baseline
import pandas as pd

score_table = score_comparison_table(teacher_summary, student_summary, min_diff=MIN_SCORE_DIFF, ascending=False)
print(f"{len(score_table)} rows")
with pd.option_context("display.max_rows", len(score_table)):
    display(score_table)